# Notebook 3 — Grad-CAM Explainability

This notebook implements **Grad-CAM (Gradient-weighted Class Activation Mapping)** for the selected ResNet50 transfer-learning model.

**Apple Image → ResNet50 → Prediction + Confidence → Grad-CAM → Heatmap → Visual Explanation**

Four classes:
- 0 — Apple Healthy
- 1 — Apple Scab
- 2 — Apple Black Rot
- 3 — Apple Cedar Apple Rust

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Project paths and configuration

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/XAI_Project')
DATA_DIR = PROJECT_DIR / 'processed'
MODEL_DIR = PROJECT_DIR / 'saved_models'
RESULTS_DIR = PROJECT_DIR / 'results'
GRADCAM_DIR = RESULTS_DIR / 'gradcam'
GRADCAM_DIR.mkdir(parents=True, exist_ok=True)
HEATMAP_DIR = GRADCAM_DIR / 'heatmaps'
OVERLAY_DIR = GRADCAM_DIR / 'overlays'
HEATMAP_DIR.mkdir(parents=True, exist_ok=True)
OVERLAY_DIR.mkdir(parents=True, exist_ok=True)

TEST_PATH = DATA_DIR / 'test.pt'
PRIMARY_CHECKPOINT = MODEL_DIR / 'best_overall_model.pt'
FALLBACK_CHECKPOINT = MODEL_DIR / 'resnet50_best.pt'

CLASS_NAMES = [
    'Apple Healthy',
    'Apple Scab',
    'Apple Black Rot',
    'Apple Cedar Apple Rust',
]
NUM_CLASSES = 4
IMAGE_SIZE = (224, 224)

print('Test data:', TEST_PATH)
print('Primary checkpoint:', PRIMARY_CHECKPOINT)
print('Grad-CAM output:', GRADCAM_DIR)
print('Classes:', CLASS_NAMES)
print('Heatmaps:', HEATMAP_DIR)
print('Overlays:', OVERLAY_DIR)

## 3. Import libraries

In [ ]:
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torchvision.models import resnet50
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 4. Load the saved ResNet50 checkpoint

In [ ]:
if PRIMARY_CHECKPOINT.exists():
    CHECKPOINT_PATH = PRIMARY_CHECKPOINT
elif FALLBACK_CHECKPOINT.exists():
    CHECKPOINT_PATH = FALLBACK_CHECKPOINT
else:
    raise FileNotFoundError(
        'No ResNet50 checkpoint found. Expected:\n'
        f'{PRIMARY_CHECKPOINT}\n{FALLBACK_CHECKPOINT}'
    )

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location='cpu',
    weights_only=False,
)

saved_class_names = checkpoint.get('class_names', CLASS_NAMES)
saved_num_classes = checkpoint.get('num_classes', len(saved_class_names))

print('Loaded:', CHECKPOINT_PATH)
print('Checkpoint keys:', list(checkpoint.keys()))
print('Saved classes:', saved_class_names)

if list(saved_class_names) != CLASS_NAMES:
    raise ValueError('Checkpoint class mapping does not match the 4 Apple classes.')
if saved_num_classes != NUM_CLASSES:
    raise ValueError(f'Checkpoint has {saved_num_classes} classes, expected {NUM_CLASSES}.')

## 5. Rebuild ResNet50 and load trained weights

In [ ]:
model = resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print('Final classifier:', model.fc)
print('ResNet50 loaded successfully.')

## 6. Load the processed Apple test set

In [ ]:
def load_split(path):
    try:
        saved_data = torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        saved_data = torch.load(path, map_location='cpu')

    if isinstance(saved_data, Dataset):
        return saved_data

    if isinstance(saved_data, dict):
        images = saved_data.get('images', saved_data.get('X'))
        labels = saved_data.get('labels', saved_data.get('y'))
    else:
        images, labels = saved_data[0], saved_data[1]

    images = torch.as_tensor(images).float()
    labels = torch.as_tensor(labels).long().view(-1)

    if images.ndim == 4 and images.shape[1] not in (1, 3):
        if images.shape[-1] in (1, 3):
            images = images.permute(0, 3, 1, 2).contiguous()

    return TensorDataset(images, labels)

test_data = load_split(TEST_PATH)

print('Test images:', len(test_data))
print('Image shape:', test_data[0][0].shape)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

## 7. Check test-set class distribution

In [ ]:
test_labels = [int(test_data[i][1]) for i in range(len(test_data))]

for class_id, class_name in enumerate(CLASS_NAMES):
    print(f'{class_id}: {class_name} -> {test_labels.count(class_id)} images')

## 8. ImageNet denormalization

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def denormalize_image(tensor):
    image = tensor.detach().cpu().numpy()
    image = np.transpose(image, (1, 2, 0))
    image = image * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(image, 0, 1)

## 9. Prediction function

In [ ]:
def predict_image(image_tensor):
    model.eval()
    image_batch = image_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(image_batch)
        probabilities = torch.softmax(logits, dim=1)

    confidence, predicted_class = probabilities.max(dim=1)

    return {
        'predicted_class': int(predicted_class.item()),
        'confidence': float(confidence.item()),
        'probabilities': probabilities[0].cpu().numpy(),
        'logits': logits[0].cpu().numpy(),
    }

## 10. Test one prediction

In [ ]:
sample_index = 0
sample_image, sample_label = test_data[sample_index]
prediction = predict_image(sample_image)

print('True class:', CLASS_NAMES[int(sample_label)])
print('Predicted class:', CLASS_NAMES[prediction['predicted_class']])
print(f"Confidence: {prediction['confidence'] * 100:.2f}%")

for class_id, probability in enumerate(prediction['probabilities']):
    print(f'{CLASS_NAMES[class_id]}: {probability * 100:.2f}%')

plt.figure(figsize=(5, 5))
plt.imshow(denormalize_image(sample_image))
plt.title(
    f"True: {CLASS_NAMES[int(sample_label)]}\n"
    f"Predicted: {CLASS_NAMES[prediction['predicted_class']]} "
    f"({prediction['confidence'] * 100:.2f}%)"
)
plt.axis('off')
plt.show()

## 11. Grad-CAM implementation

For ResNet50, Grad-CAM uses the final convolutional block `model.layer4[-1]`.

The target-class score is backpropagated to the feature maps. The gradients are globally averaged to obtain channel weights, which are then used to form the class activation map.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        self.forward_handle = target_layer.register_forward_hook(
            self._save_activations
        )
        self.backward_handle = target_layer.register_full_backward_hook(
            self._save_gradients
        )

    def _save_activations(self, module, inputs, output):
        self.activations = output

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, image_tensor, target_class=None):
        self.model.eval()
        self.model.zero_grad(set_to_none=True)

        image_batch = image_tensor.unsqueeze(0).to(device)
        image_batch.requires_grad_(True)

        logits = self.model(image_batch)
        probabilities = torch.softmax(logits, dim=1)

        if target_class is None:
            target_class = int(logits.argmax(dim=1).item())

        logits[0, target_class].backward()

        activations = self.activations[0]
        gradients = self.gradients[0]

        weights = gradients.mean(dim=(1, 2), keepdim=True)
        cam = torch.relu((weights * activations).sum(dim=0))

        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()

        return (
            cam.detach().cpu().numpy(),
            logits[0].detach().cpu().numpy(),
            probabilities[0].detach().cpu().numpy(),
            target_class,
        )

    def close(self):
        self.forward_handle.remove()
        self.backward_handle.remove()

target_layer = model.layer4[-1]
gradcam = GradCAM(model, target_layer)

print('Grad-CAM target layer: model.layer4[-1]')

## 12. Create heatmap and overlay

In [ ]:
def resize_heatmap(heatmap, target_size=(224, 224)):
    heatmap_uint8 = np.uint8(255 * heatmap)
    heatmap_image = Image.fromarray(heatmap_uint8)
    heatmap_image = heatmap_image.resize(
        target_size,
        Image.Resampling.BILINEAR,
    )
    return np.asarray(heatmap_image).astype(np.float32) / 255.0

def create_gradcam_overlay(image_tensor, heatmap, alpha=0.45):
    original = denormalize_image(image_tensor)
    heatmap_resized = resize_heatmap(
        heatmap,
        (original.shape[1], original.shape[0]),
    )

    heatmap_rgb = plt.get_cmap('jet')(heatmap_resized)[..., :3]
    overlay = np.clip(
        (1 - alpha) * original + alpha * heatmap_rgb,
        0,
        1,
    )

    return original, heatmap_resized, overlay

## 13. Main Grad-CAM visualization

In [ ]:
sample_index = 0
sample_image, true_label = test_data[sample_index]

prediction = predict_image(sample_image)

heatmap, logits, probabilities, target_class = gradcam.generate(
    sample_image,
    target_class=prediction['predicted_class'],
)

original, heatmap_resized, overlay = create_gradcam_overlay(
    sample_image,
    heatmap,
    alpha=0.45,
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(original)
axes[0].set_title(f'Original\nTrue: {CLASS_NAMES[int(true_label)]}')

axes[1].imshow(heatmap_resized, cmap='jet')
axes[1].set_title(f'Grad-CAM\nTarget: {CLASS_NAMES[target_class]}')

axes[2].imshow(overlay)
axes[2].set_title(
    f'Overlay\nPrediction: {CLASS_NAMES[target_class]}\n'
    f'Confidence: {probabilities[target_class] * 100:.2f}%'
)

for ax in axes:
    ax.axis('off')

plt.tight_layout()

single_output = GRADCAM_DIR / 'gradcam_single_example.png'
plt.savefig(single_output, dpi=200, bbox_inches='tight')
plt.show()

print('Saved:', single_output)

## 14. Grad-CAM examples for all four Apple classes

In [ ]:
def find_correct_example_for_class(dataset, class_id):
    for index in range(len(dataset)):
        image, label = dataset[index]
        if int(label) != class_id:
            continue

        prediction = predict_image(image)
        if prediction['predicted_class'] == class_id:
            return index
    return None

class_example_indices = {
    class_id: find_correct_example_for_class(test_data, class_id)
    for class_id in range(NUM_CLASSES)
}

print('Example indices:')
for class_id, index in class_example_indices.items():
    print(f'{CLASS_NAMES[class_id]} -> {index}')

In [ ]:
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(15, 5 * NUM_CLASSES))

for class_id in range(NUM_CLASSES):
    index = class_example_indices[class_id]

    if index is None:
        for col in range(3):
            axes[class_id, col].axis('off')
        axes[class_id, 1].set_title(
            f'No correctly classified example: {CLASS_NAMES[class_id]}'
        )
        continue

    image, true_label = test_data[index]
    prediction = predict_image(image)

    heatmap, logits, probabilities, target_class = gradcam.generate(
        image,
        target_class=prediction['predicted_class'],
    )

    original, heatmap_resized, overlay = create_gradcam_overlay(
        image,
        heatmap,
        alpha=0.45,
    )

    axes[class_id, 0].imshow(original)
    axes[class_id, 0].set_title(
        f'Original\nTrue: {CLASS_NAMES[int(true_label)]}'
    )

    axes[class_id, 1].imshow(heatmap_resized, cmap='jet')
    axes[class_id, 1].set_title(
        f'Grad-CAM\nTarget: {CLASS_NAMES[target_class]}'
    )

    axes[class_id, 2].imshow(overlay)
    axes[class_id, 2].set_title(
        f'Overlay\nConfidence: {probabilities[target_class] * 100:.2f}%'
    )

    for col in range(3):
        axes[class_id, col].axis('off')

plt.tight_layout()

all_classes_output = GRADCAM_DIR / 'gradcam_all_classes.png'
plt.savefig(all_classes_output, dpi=200, bbox_inches='tight')
plt.show()

print('Saved:', all_classes_output)

## 15. Grad-CAM test grid

In [ ]:
def make_gradcam_grid(indices, output_name='gradcam_test_grid.png'):
    n = len(indices)
    fig, axes = plt.subplots(n, 3, figsize=(15, 4.5 * n))

    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, index in enumerate(indices):
        image, true_label = test_data[index]
        prediction = predict_image(image)

        heatmap, logits, probabilities, target_class = gradcam.generate(
            image,
            target_class=prediction['predicted_class'],
        )

        original, heatmap_resized, overlay = create_gradcam_overlay(
            image,
            heatmap,
            alpha=0.45,
        )

        axes[row, 0].imshow(original)
        axes[row, 0].set_title(f'True: {CLASS_NAMES[int(true_label)]}')

        axes[row, 1].imshow(heatmap_resized, cmap='jet')
        axes[row, 1].set_title(f'Grad-CAM: {CLASS_NAMES[target_class]}')

        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title(
            f'Confidence: {probabilities[target_class] * 100:.2f}%'
        )

        for col in range(3):
            axes[row, col].axis('off')

    plt.tight_layout()
    output_path = GRADCAM_DIR / output_name
    plt.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.show()
    return output_path

rng = random.Random(SEED)
grid_indices = rng.sample(range(len(test_data)), min(6, len(test_data)))

grid_output = make_gradcam_grid(grid_indices)
print('Saved:', grid_output)

## 16. Quantitative Grad-CAM statistics

The dataset does not contain pixel-level disease localization masks, so **IoU is not treated as a ground-truth Grad-CAM metric** here.

Instead, we record descriptive activation statistics:
- maximum activation
- mean activation
- percentage of pixels above a high-activation threshold

These describe attention concentration; they do not measure localization accuracy.

In [ ]:
def gradcam_statistics(heatmap, threshold=0.5):
    heatmap_resized = resize_heatmap(heatmap, IMAGE_SIZE)

    return {
        'gradcam_max': float(np.max(heatmap_resized)),
        'gradcam_mean': float(np.mean(heatmap_resized)),
        'high_activation_area_percent': float(
            np.mean(heatmap_resized >= threshold) * 100
        ),
    }

print('Grad-CAM statistics function ready.')

## 17. Prediction + explanation report for the complete test set

In [ ]:
report_rows = []

print(f'Generating and saving Grad-CAM for all {len(test_data)} test images...')
print(f'Heatmaps will be saved to: {HEATMAP_DIR}')
print(f'Overlays will be saved to: {OVERLAY_DIR}')

for index in range(len(test_data)):
    image, true_label = test_data[index]
    prediction = predict_image(image)

    heatmap, logits, probabilities, target_class = gradcam.generate(
        image,
        target_class=prediction['predicted_class'],
    )

    original, heatmap_resized, overlay = create_gradcam_overlay(
        image,
        heatmap,
        alpha=0.45,
    )

    # Save the raw normalized Grad-CAM heatmap as a color image.
    heatmap_uint8 = np.uint8(np.clip(heatmap_resized, 0, 1) * 255)
    heatmap_rgb = plt.get_cmap('jet')(heatmap_uint8 / 255.0)[..., :3]
    heatmap_rgb_uint8 = np.uint8(np.clip(heatmap_rgb, 0, 1) * 255)

    heatmap_path = HEATMAP_DIR / f'image_{index:05d}.png'
    overlay_path = OVERLAY_DIR / f'image_{index:05d}.png'

    Image.fromarray(heatmap_rgb_uint8).save(heatmap_path)
    Image.fromarray(np.uint8(np.clip(overlay, 0, 1) * 255)).save(overlay_path)

    row = {
        'index': index,
        'heatmap_path': str(heatmap_path),
        'overlay_path': str(overlay_path),
        'true_label_id': int(true_label),
        'true_class': CLASS_NAMES[int(true_label)],
        'predicted_label_id': target_class,
        'predicted_class': CLASS_NAMES[target_class],
        'confidence': float(probabilities[target_class]),
        'correct': int(target_class == int(true_label)),
        **gradcam_statistics(heatmap),
    }

    for class_id, class_name in enumerate(CLASS_NAMES):
        safe_name = class_name.replace(' ', '_')
        row[f'prob_{class_id}_{safe_name}'] = float(probabilities[class_id])

    report_rows.append(row)

    if (index + 1) % 100 == 0 or index == len(test_data) - 1:
        print(f'Processed {index + 1}/{len(test_data)} images')

explanation_report = pd.DataFrame(report_rows)
report_output = RESULTS_DIR / 'gradcam_prediction_report.csv'
explanation_report.to_csv(report_output, index=False)

display(explanation_report.head())
print('Saved:', report_output)
print('Total heatmaps saved:', len(list(HEATMAP_DIR.glob('*.png'))))
print('Total overlays saved:', len(list(OVERLAY_DIR.glob('*.png'))))

## 18. Test-set classification report

In [ ]:
y_true = []
y_pred = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        logits = model(images)
        predictions = logits.argmax(dim=1)

        y_true.extend(labels.tolist())
        y_pred.extend(predictions.cpu().tolist())

print(
    classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
)

cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
display(cm_df)

## 19. Save Grad-CAM configuration

In [ ]:
gradcam_config = {
    'model': 'ResNet50',
    'checkpoint': str(CHECKPOINT_PATH),
    'target_layer': 'model.layer4[-1]',
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'input_size': list(IMAGE_SIZE),
    'normalization': {
        'mean': [0.485, 0.456, 0.406],
        'std': [0.229, 0.224, 0.225],
    },
    'overlay_alpha': 0.45,
    'outputs': {
        'heatmaps': str(HEATMAP_DIR),
        'overlays': str(OVERLAY_DIR),
        'prediction_report': str(RESULTS_DIR / 'gradcam_prediction_report.csv'),
    },
    'dataset': {
        'name': 'PlantVillage',
        'subset': 'Apple',
        'total_images': 7771,
        'test_images_processed_for_gradcam': len(test_data),
    },
}

config_output = RESULTS_DIR / 'gradcam_config.json'

with open(config_output, 'w') as f:
    json.dump(gradcam_config, f, indent=4)

print('Saved:', config_output)

# Notebook 3 — Grad-CAM Explainability

## 1. Objective

The main objective of this notebook is to add **Explainable AI (XAI)** to our Apple plant disease classification system.

In the previous notebook, we trained and compared different transfer-learning models:

- ResNet50
- EfficientNet-B0
- MobileNetV3-Large

Based on the evaluation results, **ResNet50 was selected as the final classification model**.

However, a classification model only tells us:

> **"What disease does the leaf have?"**

It does not directly tell us:

> **"Which regions of the leaf influenced the model's prediction?"**

Therefore, in this notebook we use **Grad-CAM (Gradient-weighted Class Activation Mapping)** to visualize the regions of the Apple leaf that contributed to the model's prediction.

---

# 2. Dataset Used

This notebook uses the Apple subset of the PlantVillage dataset.

The dataset contains **7,771 Apple leaf images** belonging to four classes:

| Label | Class | Images |
|---|---|---:|
| 0 | Apple Healthy | 2,008 |
| 1 | Apple Scab | 2,016 |
| 2 | Apple Black Rot | 1,987 |
| 3 | Apple Cedar Apple Rust | 1,760 |
| | **Total** | **7,771** |

The dataset was divided into:

```text
70% → Training
15% → Validation
15% → Testing
```

This notebook primarily works with the **test set**, because the test set represents images that were not used during model training.

The test set contains approximately **1,166 images**.

---

# 3. Input to This Notebook

The main inputs are:

### 1. Trained ResNet50 model

The best trained ResNet50 model obtained from the transfer-learning notebook is loaded from Google Drive.

The model has been modified from its original ImageNet classification task to classify four Apple leaf classes.

The final output layer contains:

```text
4 output neurons
```

corresponding to:

```text
0 → Apple Healthy
1 → Apple Scab
2 → Apple Black Rot
3 → Apple Cedar Apple Rust
```

### 2. Test dataset

The processed test dataset is loaded from:

```text
XAI_Project/processed/test.pt
```

Each image has already been:

```text
Resized → 224 × 224
Converted → RGB
Normalized → ImageNet normalization
```

The tensor format is:

```text
3 × 224 × 224
```

For a single image with a batch dimension:

```text
1 × 3 × 224 × 224
```

### 3. Grad-CAM configuration

The notebook uses the following configuration:

```text
Target model → ResNet50
Target layer → model.layer4[-1]
Image size → 224 × 224
Normalization → ImageNet
Number of classes → 4
```

---

# 4. Why ResNet50 Is Used

ResNet50 was selected as the final model after comparing the transfer-learning models.

The model was originally trained on ImageNet and contains useful visual features learned from a large image dataset.

For our project, the original classification layer was replaced with a new layer capable of predicting the four Apple disease classes.

Therefore:

```text
Image
   ↓
ResNet50 convolutional layers
   ↓
Feature extraction
   ↓
Final classification layer
   ↓
4 Apple disease classes
```

---

# 5. Prediction Process

For each test image, the ResNet50 model performs a forward pass.

The model produces four raw class scores called **logits**.

These logits are converted into probabilities using the Softmax function.

Conceptually:

```text
Input Apple Leaf
       ↓
ResNet50
       ↓
4 Class Logits
       ↓
Softmax
       ↓
Class Probabilities
       ↓
Predicted Class
```

For example:

```text
Apple Healthy            → 0.01
Apple Scab               → 0.02
Apple Black Rot          → 0.03
Apple Cedar Apple Rust   → 0.94
```

The predicted class is the class having the highest probability.

The corresponding probability is used as the model's prediction confidence.

---

# 6. What Is Grad-CAM?

**Grad-CAM** stands for:

> **Gradient-weighted Class Activation Mapping**

Grad-CAM is an explainability technique for deep learning models, particularly convolutional neural networks.

It helps us understand **which spatial regions of an image contributed to a particular prediction**.

For example, instead of only saying:

```text
Prediction: Apple Scab
Confidence: 97%
```

Grad-CAM allows us to additionally visualize:

```text
The regions of the leaf that contributed strongly
to the Apple Scab prediction.
```

This makes the model more interpretable.

---

# 7. Grad-CAM Target Layer

For ResNet50, Grad-CAM is applied to:

```python
model.layer4[-1]
```

This is the final convolutional block of ResNet50.

The reason for using a late convolutional layer is that it contains high-level visual features while still retaining spatial information.

Earlier layers generally capture features such as:

```text
Edges
Textures
Simple patterns
Colors
```

Later convolutional layers capture more complex visual patterns.

For a plant disease classification task, these patterns can potentially correspond to visual characteristics such as:

```text
Leaf spots
Lesions
Discoloration
Disease-related textures
Affected leaf regions
```

Therefore, the final convolutional layer is suitable for generating Grad-CAM explanations.

---

# 8. Grad-CAM Working Process

The Grad-CAM process used in this notebook can be summarized as follows.

### Step 1 — Forward Pass

The Apple leaf image is passed through ResNet50.

```text
Apple Leaf
     ↓
ResNet50
     ↓
Class Scores
```

### Step 2 — Select Target Class

The predicted class is selected as the target class for explanation.

For example:

```text
Target Class = Apple Scab
```

### Step 3 — Backpropagation

The gradient of the target class score is calculated with respect to the feature maps of:

```text
model.layer4[-1]
```

### Step 4 — Calculate Importance Weights

The gradients are averaged spatially to obtain weights representing the importance of each feature map for the selected class.

### Step 5 — Combine Feature Maps

The feature maps are weighted using these gradient-based importance values and combined.

### Step 6 — Apply ReLU

ReLU is applied so that the Grad-CAM focuses on positive contributions to the selected class.

### Step 7 — Normalize

The resulting activation map is normalized so that its values lie approximately between:

```text
0 and 1
```

### Step 8 — Resize

The Grad-CAM heatmap is resized to:

```text
224 × 224
```

so that it matches the input image.

### Step 9 — Overlay

The heatmap is placed over the original image to create an interpretable visualization.

---

# 9. Grad-CAM Output

For each image, the main visual outputs are:

### Original Image

The original Apple leaf image.

### Grad-CAM Heatmap

A heatmap representing the spatial regions that contributed to the selected prediction.

Generally:

```text
Higher activation → stronger contribution
Lower activation → weaker contribution
```

### Grad-CAM Overlay

The heatmap is combined with the original leaf image.

This allows us to visually inspect whether the model is focusing on meaningful parts of the leaf.

---

# 10. Grad-CAM for the Complete Test Set

Instead of applying Grad-CAM to only a few demonstration images, this project applies Grad-CAM to the **complete test set**.

Approximately:

```text
1,166 test images
```

are processed.

For each image:

```text
Test Image
    ↓
ResNet50 Prediction
    ↓
Predicted Class
    ↓
Confidence
    ↓
Grad-CAM
    ↓
Heatmap
    ↓
Overlay
    ↓
Save Results
```

This allows us to perform a more complete XAI analysis rather than relying only on a few selected examples.

---

# 11. Saved Grad-CAM Images

The Grad-CAM outputs are saved directly to Google Drive.

The directory structure is:

```text
XAI_Project/
└── results/
    └── gradcam/
        ├── heatmaps/
        │   ├── image_00000.png
        │   ├── image_00001.png
        │   ├── image_00002.png
        │   └── ...
        │
        └── overlays/
            ├── image_00000.png
            ├── image_00001.png
            ├── image_00002.png
            └── ...
```

The complete test set can therefore be inspected later without rerunning the entire Grad-CAM generation process.

---

# 12. Important Difference Between Model Weights and Grad-CAM

The trained ResNet50 model has permanent model weights stored in the model checkpoint.

Grad-CAM itself does **not** have a separate permanent set of model weights.

For each image, Grad-CAM calculates gradient-based importance values dynamically.

Therefore, we save:

```text
ResNet50 model weights
+
Grad-CAM configuration
+
Generated heatmaps
+
Generated overlays
+
Prediction information
+
Grad-CAM statistics
```

The Grad-CAM configuration ensures that the same procedure can be reproduced later.

---

# 13. Grad-CAM Configuration

The configuration contains information such as:

```text
Model → ResNet50
Target layer → layer4[-1]
Image size → 224 × 224
Normalization mean → ImageNet
Normalization standard deviation → ImageNet
Class mapping → Four Apple classes
```

This configuration will be useful when integrating Grad-CAM into the future web application.

---

# 14. Prediction Report

For every test image, prediction information is recorded.

The report contains information such as:

```text
Image ID
True Class
Predicted Class
Confidence
Class Probabilities
Correct / Incorrect Prediction
```

This allows us to compare the model's prediction with the actual test label.

---

# 15. Grad-CAM Statistics

The notebook also calculates descriptive statistics from the Grad-CAM maps.

Examples include:

```text
Maximum activation
Mean activation
High-activation area percentage
```

These statistics help us quantify the distribution of the model's visual attention.

They are **descriptive statistics**, not a direct measurement of disease localization accuracy.

---

# 16. Why We Do Not Calculate IoU

Grad-CAM produces an explanation heatmap, but our dataset does not contain pixel-level disease segmentation masks or bounding-box annotations.

Therefore, we do not have ground-truth disease regions against which the Grad-CAM map could be directly compared.

Because of this, **IoU is not used as a Grad-CAM localization metric in this notebook**.

The Grad-CAM results are instead used for visual and descriptive analysis.

---

# 17. Representative Visualizations

Although Grad-CAM is generated for the complete test set, the notebook does not display all approximately 1,166 images inside the notebook.

Instead, representative examples are displayed for easier qualitative analysis.

Examples are selected from the four classes:

```text
Apple Healthy
Apple Scab
Apple Black Rot
Apple Cedar Apple Rust
```

This keeps the notebook readable while still allowing the complete test set to be processed and saved.

---

# 18. What Are We Trying to Achieve?

The main goal of this notebook is to move our system from a **black-box classifier** toward an **explainable disease detection system**.

Without XAI:

```text
Image
  ↓
Model
  ↓
Apple Scab
```

The user only receives the prediction.

With Grad-CAM:

```text
Image
  ↓
ResNet50
  ↓
Prediction + Confidence
  ↓
Grad-CAM
  ↓
Important Image Regions
```

Therefore, the system can provide both:

```text
WHAT → Predicted disease

WHY → Regions that influenced the prediction
```

This is an important component of our overall Explainable AI project.

---

# 19. How This Helps the Next Stages of the Project

Grad-CAM is not the final component of our system.

It becomes an input for the later stages of the project.

Our planned architecture is:

```text
                 Apple Leaf Image
                         ↓
                  Preprocessing
                         ↓
                    ResNet50
                         ↓
             ┌───────────┴───────────┐
             ↓                       ↓
        Prediction                Grad-CAM
        Confidence               Explanation
             ↓                       ↓
             └───────────┬───────────┘
                         ↓
                Trust / Uncertainty
                         ↓
                       RAG
                         ↓
                       LLM
                         ↓
              Human-in-the-Loop
                         ↓
                 Feedback / Memory
                         ↓
                  Web Dashboard
```

---

# 20. Role of Grad-CAM in the Trust System

The prediction confidence alone does not explain where the model is looking.

Grad-CAM provides an additional explanation signal.

For example:

```text
Prediction:
Apple Scab

Confidence:
95%

Grad-CAM:
Strong activation concentrated on leaf regions
```

This information can later be incorporated into the project's trust and uncertainty analysis.

The important point is that **Grad-CAM is an explanation signal, not by itself a guarantee that the prediction is correct**.

---

# 21. Role in RAG and LLM

The next stage of the project will use **Retrieval-Augmented Generation (RAG)** to retrieve reliable plant-disease information.

The information flow can become:

```text
ResNet50
    ↓
Predicted Disease
    +
Confidence
    +
Grad-CAM Explanation
    ↓
RAG
    ↓
Relevant Disease Knowledge
    ↓
LLM
    ↓
Human-readable Explanation
```

The LLM can then explain the prediction using retrieved domain knowledge rather than relying only on its general pretrained knowledge.

---

# 22. Role in Human-in-the-Loop

The final system is designed to include a human in the decision process.

If the system is uncertain or the explanation is not satisfactory, the user can provide feedback.

Conceptually:

```text
Model Prediction
      ↓
Confidence + Explanation
      ↓
Trust Evaluation
      ↓
High Trust → Provide Result
      ↓
Low / Uncertain → Request Human Review
      ↓
Human Feedback
      ↓
Store Feedback
```

The stored feedback can later be used as part of the project's feedback/memory mechanism.

---

# 23. Role in the Future Web Dashboard

The Grad-CAM implementation will also be integrated into the final web dashboard.

For a new image uploaded by the user, the dashboard can perform:

```text
User Uploads Image
        ↓
Preprocessing
        ↓
ResNet50
        ↓
Prediction
        ↓
Confidence
        ↓
Grad-CAM
        ↓
Heatmap + Overlay
        ↓
Trust / Uncertainty
        ↓
RAG
        ↓
LLM Explanation
        ↓
Final Dashboard
```

The dashboard can display:

```text
Predicted Disease
Confidence
Original Image
Grad-CAM Overlay
Explanation
Trust / Uncertainty Information
Retrieved Disease Information
```

For images already processed in the test set, the saved Grad-CAM outputs can also be reused.

For **new user-uploaded images**, Grad-CAM will be generated dynamically using the same configuration.

---

# 24. Final Output of Notebook 3

The main outputs of this notebook are:

```text
1. Trained ResNet50 model
       ↓
2. Predictions for test images
       ↓
3. Prediction confidence
       ↓
4. Grad-CAM heatmaps
       ↓
5. Grad-CAM overlays
       ↓
6. Grad-CAM statistics
       ↓
7. Prediction report
       ↓
8. Grad-CAM configuration
```

These outputs are stored in Google Drive for later use.

---

# 25. Overall Conclusion

Notebook 3 adds the **Explainability Layer** to our Apple plant disease detection system.

The trained ResNet50 model is used to classify Apple leaves into four categories:

```text
Apple Healthy
Apple Scab
Apple Black Rot
Apple Cedar Apple Rust
```

Grad-CAM is then applied to the final convolutional layer of ResNet50 to identify the image regions that contributed to each prediction.

Instead of only producing a disease label and confidence score, the system now also produces a visual explanation of the prediction.

Grad-CAM is generated across the complete test set and the resulting heatmaps and overlays are saved to Google Drive. Representative examples are displayed in the notebook for qualitative analysis.

This notebook therefore establishes the **XAI component of the overall project** and provides the foundation for the next stages:

```text
Grad-CAM
    ↓
Trust / Uncertainty
    ↓
RAG
    ↓
LLM-based Explanation
    ↓
Human-in-the-Loop
    ↓
Feedback / Memory
    ↓
Web Dashboard
```

The overall goal is to develop a system that does not simply predict an Apple leaf disease, but provides a **prediction, confidence, visual explanation, supporting knowledge, and a mechanism for human feedback**.